# [1교시]

# 1 — let/const, 화살표 함수, 템플릿 리터럴

## 1. 학습 목표
* 기존 `var` 키워드가 가진 구조적 문제점(호이스팅, 재선언 허용, 함수 레벨 스코프)을 분석하고, 현대 JavaScript 표준인 `let`과 `const`가 제공하는 블록 레벨 스코프와 TDZ(Temporal Dead Zone) 안전망의 중요성을 파악합니다.
* 루프 내에서의 변수 참조 누수 및 객체 가변성 제어 기법인 `Object.freeze()`의 동작 원리를 이해합니다.
* 화살표 함수(Arrow Function)의 매개변수 개수에 따른 단축 문법과 일반 함수와의 치명적 격차인 `this` 바인딩(렉시컬 스코프 기반 `this` 상속)의 작동 메커니즘을 마스터합니다.
* 템플릿 리터럴(Template Literals)을 활용한 다줄(Multi-line) 문자열 표현 및 내부 표현식 삼항 연산자를 연계한 조건부 HTML 렌더링 패턴을 학습합니다.

---

## 2. var vs let / const 심화 분석

### 1) var 변수의 한계와 문제 사례
* **재선언 남용** : 동일 스코프 내에서 동일 변수명을 아무런 제약 없이 재선언하여 값을 덮어쓸 수 있었기 때문에 협업 과정에서 기존 중요 전역/지역 변수를 파괴하는 버그가 잦았습니다.
* **함수 레벨 스코프(Function Scope)** : 블록 내부(`if`, `for` 등)에서 선언된 변수가 블록 외부로 탈출하여 영향을 미칩니다. 특히 반복문 루프 카운터로 사용된 `var`가 루프 종료 후에도 남아 메모리와 전역 스코프를 오염시켰습니다.
* **호이스팅과 초기화 분리** : 변수 선언부가 최상단으로 오르는 호이스팅으로 인해 선언 전에 변수를 참조해도 에러가 아닌 `undefined`를 반환하여 의도치 않은 논리 에러를 숨겨왔습니다.

### 2) let / const 표준 변수의 구동 원리
* **블록 레벨 스코프(Block Scope)** : 중괄호 `{}`로 구분된 영역 내부에서 선언된 변수는 오직 해당 블록 내부에서만 존재하며 외부 노출이 철저히 격리됩니다.
* **TDZ (Temporal Dead Zone, 일시적 사각지대)** : `let`과 `const` 역시 호이스팅 대상이나 초기화 단계 이전까지는 메모리 참조가 엄격하게 차단되는 TDZ 구역에 머둡니다. 이 상태에서 변수를 호출하면 예외 없이 `ReferenceError`를 발생시켜 미선언 변수 사용 실수를 코드 구동 시점에 잡아냅니다.
* **상수 객체의 가변성과 동결(Object.freeze)** : `const`로 지정된 변수 자체는 다른 메모리 주소값으로 재할당될 수 없습니다. 하지만 내부의 멤버 프로퍼티나 배열 원소는 자유롭게 변경될 수 있는 얕은 불변성을 가집니다. 완전히 값을 읽기 전용으로 가두기 위해서는 `Object.freeze()` API를 사용하여 객체를 명시적으로 동결해야 합니다.

---

## 3. 화살표 함수 (Arrow Function)의 동작과 활용

### 1) 다양한 형태의 문법 규격
화살표 함수는 함수 표현식을 아주 간결하고 우아하게 표기할 수 있는 문법 설탕을 제공합니다.
* **기본형**: `const myFunc = (a, b) => { return a + b; };`
* **암시적 반환 (Implicit Return)** : 함수의 실행 블록이 단일 연산 및 단일 반환식으로만 구성된 경우, 중괄호 `{}`와 `return` 지시자를 동시에 생략할 수 있습니다. `const myFunc = (a, b) => a + b;`
* **인자 생략 규칙** : 매개변수가 오직 한 개일 때에 한하여 소괄호 `()`를 생략할 수 있습니다. 매개변수가 아예 없거나 두 개 이상이면 반드시 소괄호로 감싸야 합니다. `const myFunc = x => x * x;`

### 2) 화살표 함수와 일반 함수의 this 비교 
this가 가리키는 대상이 결정되는 시점이 두 함수 형식에 따라 근본적으로 다릅니다.
* **일반 함수** : 함수가 **호출되는 방식** 에 따라 동적으로 바인딩됩니다. 메서드로 호출하면 호출한 주체 객체를 가리키지만, 콜백 함수로 인계되거나 일반 독립 함수로 실행되면 내부 `this`가 전역 객체(window 또는 strict mode에서는 undefined)로 지정되어 내부 정보에 정상 접근하지 못합니다.
* **화살표 함수** : 자신만의 `this` 바인딩을 아예 가지지 않으며, 함수가 **정의(작성)되는 시점의 외부 렉시컬 스코프의 `this`**를 그대로 물려받아 불변형으로 영구 고정시킵니다.
* **setTimeout 비동기  사례** : 객체 내에서 비동기 콜백을 등록할 때 일반 함수를 전달하면 내부 `this`가 끊기며 호출 오류를 내지만, 화살표 함수를 적용하면 상위 객체 메서드 스코프의 `this`가 깨끗하게 상속되어 문제없이 부모 객체의 속성을 지탱합니다.

---

## 4. 템플릿 리터럴 (Template Literals) 및 조건부 마크업

백틱(`` ` ``) 지시자를 이용하여 텍스트 구조를 기술하며, 동적 변수와 프로그래밍 코드를 우아하게 혼합하는 표준 문법입니다.
* **문자열 병합의 편리성** : 기존 플러스 기호(`+`)를 통한 조각조각의 접합을 폐기하고 `${expression}` 블록을 사용하여 텍스트와 자바스크립체 실행 결과물을 바로 치환시킵니다.
* **멀티라인 지원** : 코드 내에서 개행(엔터)을 입력하면 텍스트 자체의 포맷에 개행이 그대로 인정되어 동적 HTML 태그 구조를 여러 줄로 가독성 있게 기술하기에 탁월합니다.
* **조건부 렌더링 내장** : 템플릿 리터럴 내부 `${}` 안에는 임의의 자바스크립트 표현식이 올 수 있으므로, 삼항 연산자를 사용하여 특정 객체 정보 유무나 플래그 상태에 따라 서로 다른 HTML 조각을 안전하게 동적 생성해낼 수 있습니다. `<div>${user.isPremium ? "<span class='badge'>프리미엄</span>" : ""}</div>`

---

## 5. 실습 미션 적용 (이론 적용 코드)
다중 회원 데이터 리스트(API 응답 모사)를 획득하여 루프 스코프의 안전성을 검토하고, 화살표 함수와 조건부 템플릿 리터럴 문법을 융합하여 프리미엄 등급 여부에 따라 배지가 차등 출력되는 카드를 그리드 형태의 화면으로 구축합니다.

# [2교시] ~ [3교시]

01_es6_basic/main_01.js

# [4교시]

#  구조분해 할당 및 전개/나머지 연산자

## 1. 학습 목표
* 객체(Object)와 배열(Array)에 저장된 복잡한 요소를 단 한 줄의 선언식으로 분해하여 개별 변수에 효율적으로 바인딩하는 구조분해 할당(Destructuring Assignment)의 고급 용법을 습득합니다.
* 중첩 구조분해 할당 및 함수의 매개변수 구조분해 할당 패턴을 이해하여 코드 가독성을 극대화합니다.
* 전개 연산자(Spread Operator, `...`)의 메모리 얕은 복사(Shallow Copy) 동작으로 발생하는 데이터 동시 오염 버그를 분석하고, 이를 완전히 차단하는 깊은 복사(Deep Copy) 구현 기법을 학습합니다.
* 가변 매개변수를 수거하는 나머지 매개변수(Rest Parameter, `...rest`)를 활용해 매개변수의 유연성을 확보하고 배열 고차 함수와 연계하는 실무 기법을 파악합니다.

---

## 2. 구조분해 할당 (Destructuring Assignment) 고도화

### 1) 객체 구조분해 할당의 실무적 응용
* **기본값과 변수명 우회 변경**:
  ```javascript
  const member = { id: "user01", nick: "초보개발자" };
  // nick 속성을 새로운 변수명 userNick으로 꺼내고, 누락된 role 속성에는 기본값 'guest'를 할당합니다.
  const { nick: userNick, role = "guest" } = member;
  ```
* **중첩 객체(Nested Object) 분해**:
  객체 내부에 또 다른 객체가 존재하는 경우, 하위 깊이까지 단 한 번의 구조분해 선언으로 도달할 수 있습니다.
  ```javascript
  const client = { name: "최영희", info: { email: "young@mail.com", addr: "서울" } };
  const { info: { email } } = client; // client.info.email 값이 바로 email 변수에 바인딩됩니다.
  ```
* **함수 매개변수 구조분해**:
  함수의 인자로 객체를 넘길 때 매개변수 정의단에서 즉시 분해하여 활용하면 함수 내부의 임시 참조 과정을 대폭 줄일 수 있습니다.
  ```javascript
  const printProfile = ({ name, info: { email } }) => {
    console.log(`이름: ${name}, 이메일: ${email}`);
  };
  ```

### 2) 배열 구조분해 할당의 응용
* **변수 값 맞교환 (Swap)**: 임시 변수(`temp`)를 생성하지 않고 단 한 줄로 두 변수의 값을 서로 스왑할 수 있습니다.
  ```javascript
  let valueA = 10;
  let valueB = 20;
  [valueA, valueB] = [valueB, valueA]; // valueA는 20, valueB는 10이 됨
  ```
* **건너뛰기 및 나머지 수거**: 콤마를 통해 필요 없는 자리는 뛰어넘고, 남은 원소는 나머지 기호(`...rest`)로 배열 수거가 가능합니다.
  ```javascript
  const numbers = [1, 2, 3, 4, 5];
  const [first, , third, ...rest] = numbers; // first = 1, third = 3, rest = [4, 5]
  ```

---

## 3. 전개 연산자 (Spread Operator)와 데이터 얕은 복사 오염 실증

### 1) 얕은 복사 (Shallow Copy)의 동작과 버그
전개 연산자(`...`)로 배열이나 객체를 복사하면 1단계 깊이에 있는 원소들은 온전하게 복사되지만, 2단계 이상의 깊이에 포함된 중첩 객체나 배열은 메모리상의 주소(Reference)를 복사 사본과 공유하게 됩니다.
이로 인해 복사된 사본의 내부 중첩 속성을 수정하면 **원본 객체의 내부 속성까지 함께 변경되어 버리는 연쇄 오염 버그**가 생깁니다.

### 2) 깊은 복사 (Deep Copy)를 통한 안전 지대 확보
중첩된 구조의 데이터도 오염 없이 완전히 분리하기 위해 깊은 복사를 수행해야 합니다.
* **수동 중첩 복사**: 전개 연산자를 여러 깊이에서 재귀적으로 중첩해서 사용하는 방식입니다.
  ```javascript
  const copied = { ...original, detail: { ...original.detail } };
  ```
* **structuredClone API**: 최신 브라우저 표준 내장 API로, 복잡한 객체와 배열의 중첩 참조를 완벽하게 분리하여 완벽한 사본(Deep Copy)을 만들어 줍니다.
  ```javascript
  const deepCopied = structuredClone(original);
  ```

---

## 4. 나머지 매개변수 (Rest Parameter)

함수 시그니처의 마지막 매개변수로 작성하면, 지정된 매개변수들을 넘어서서 전달된 남은 모든 인수(Arguments)를 하나의 순수 배열로 수집합니다.
* **배열 API와 즉각 결합**: 가변 개수의 인수들이 배열 타입으로 바인딩되므로 `reduce`, `map`, `filter` 등의 고차 함수를 루프 없이 바로 얹어서 연산할 수 있습니다.
  ```javascript
  const filterEvens = (...numbers) => numbers.filter(n => n % 2 === 0);
  ```

---

## 5. 실습 미션 적용 (이론 적용 코드)
기본 쇼핑몰 상품 목록에 신규 상품을 병합하는 과정에서 얕은 복사 오염이 왜 생기는지 실증하고, `structuredClone` 및 매개변수 구조분해 할당을 적용하여 오염 없이 안전하게 대량의 카드 목록을 출력하는 환경을 빌드합니다.

# [5교시]

# DOM 선택과 탐색

## 1. 학습 목표
* 구식 DOM 선택 API(`getElementById`, `getElementsByClassName`)의 단점을 파악하고, 모던 표준 방식인 `querySelector`와 `querySelectorAll`의 셀렉터 조합 규칙을 습득합니다.
* 특정 요소를 기준으로 상위 부모, 하위 자식, 인접 형제 노드를 정밀하게 찾아 들어가는 DOM 탐색 API(`closest`, `children`, `parentElement`, `nextElementSibling`)를 이해합니다.
* 스타일을 직접 변경(`style.color = "red"`)하는 안 좋은 코드를 지양하고, CSS 클래스 규칙을 입혔다 떼어내는 `classList` 객체의 핵심 메서드(`add`, `remove`, `toggle`, `contains`)를 활용합니다.

---

## 2. 모던 DOM 선택 표준: querySelector / querySelectorAll
현대 JavaScript는 과거와 같이 ID용, 클래스용 선택 API를 구분하여 호출하지 않고, **CSS 선택자 문법 그대로** DOM을 지목합니다.

```javascript
// 1. 단일 요소 선택 (가장 먼저 만나는 1개의 노드만 반환)
const mainNav = document.querySelector("#main-nav");
const activeMenu = document.querySelector(".nav-list .active-item");

// 2. 다수 요소 선택 (매칭되는 모든 요소를 유사 배열 형태인 NodeList로 반환)
const cards = document.querySelectorAll(".feature-card");
const galleryImgs = document.querySelectorAll(".gallery img");
```

> [!IMPORTANT]
> `querySelectorAll`이 반환하는 **`NodeList`**는 순수 Array(배열)가 아닌 **유사 배열**입니다. 다행히 현대 브라우저 규격상 **`forEach`** 메서드는 기본 탑재되어 순회가 가능하지만, `map`, `filter`, `reduce` 등의 순수 배열 메서드는 직접 쓸 수 없습니다. 만약 순수 배열로 다루고 싶다면 전개 연산자를 사용하여 **`[...cards]`**로 배열 캐스팅 변환 과정을 거치는 것이 실무 관례입니다.

---

## 3. DOM 트리 상하좌우 탐색 (DOM Navigation)
원하는 요소를 한 번에 querySelector로 찾기 어려울 때, 또는 특정 이벤트가 일어난 타깃 요소를 기준으로 근처에 있는 다른 형제나 부모를 가리켜야 할 때 탐색 API를 동원합니다.

```
                  ┌─────────────────┐
                  │  parentElement  │  (상위 부모 노드 추적)
                  └────────┬────────┘
                           │ ▲
                           ▼ │
 ┌─────────────────────────┼─────────────────────────┐
 │ previousElementSibling  │  [ 기준 target 요소 ]   │  nextElementSibling
 └─────────────────────────┼─────────────────────────┘ (오른쪽 형제)
                           │ ▲
                           ▼ │
                  ┌────────┴────────┐
                  │    children     │  (하위 직속 자식 목록 NodeList)
                  └─────────────────┘
```

* **`parentElement`**: 기준 요소의 직속 상위 부모 요소를 참조합니다.
* **`children`**: 기준 요소의 하위 직속 자식 요소들을 HTMLCollection 형태로 반환합니다.
* **`closest("CSS선택자")` (매우 중요)**: 기준 요소 자신을 포함하여 **가장 가까운 상위 조상 요소 중 CSS선택자와 일치하는 대상**을 위로 거슬러 올라가며 찾아냅니다. 이벤트 위임 구현 시 타깃 카드를 역추적할 때 필수적으로 쓰입니다.
* **`nextElementSibling` / `previousElementSibling`**: 각각 오른쪽 인접 형제와 왼쪽 인접 형제 요소를 지목합니다.

---

## 4. 클래스 제어의 정석: classList
인라인 스타일(`el.style.backgroundColor = "black"`)을 JS에서 직접 다루면 디자인 변경 시 HTML/CSS/JS를 모두 뒤집어야 하므로 강하게 배제됩니다. 대신 CSS 파일에 디자인 클래스를 마련해두고, JS로는 클래스명만 삽입/삭제해 줍니다.

* **`classList.add("className")`**: 특정 클래스를 추가합니다. (기존 클래스 훼손 안 함)
* **`classList.remove("className")`**: 특정 클래스를 제거합니다.
* **`classList.toggle("className")`**: 특정 클래스가 있으면 제거하고, 없으면 넣어줍니다. (토글 스위치 기능)
* **`classList.contains("className")`**: 특정 클래스가 현재 포함되어 있는지 여부를 Boolean(`true`/`false`)으로 반환합니다.

---

## 5. 실습 
게시판의 댓글(Comment) 카드 컴포넌트 내부에서 '답글 달기' 또는 '댓글 내용 강조' 동작을 할 때, 클릭이 감지된 요소를 기준으로 가장 가까운 부모 댓글 카드(`.comment-card`)를 거슬러 올라가고(`closest`), 그 부모 밑에 위치한 댓글 본문 영역과 작성자 이름을 하향 탐색(`querySelector`)하여 클래스를 입히는 탐색 직입니다.

# [6교시]

# DOM 조작 (콘텐츠 생성/수정/삭제)

## 1. 학습 목표
* DOM 요소를 동적으로 변경할 때 사용되는 `textContent`와 `innerHTML`의 차이점 및 크로스 사이트 스크립팅(XSS) 공격에 대응하는 보안 가이드를 습득합니다.
* 메모리상에 새로운 HTML 노드를 동적으로 생성하는 `createElement`와 이를 문서 트리에 주입 및 삭제하는 노드 조작 API(`appendChild`, `prepend`, `remove`)의 사용법을 익힙니다.
* 백엔드에서 받은 데이터 배열 형식(JSON 형태 등)을 `forEach` 순회와 구조분해 할당을 통해 화면 요소를 생성하고 실시간으로 그려내는 반복 렌더링(Iterative Rendering)을 마스터합니다.

---

## 2. textContent vs innerHTML (보안 가이드)

JS로 기존 태그 내부의 내용을 바꿀 때 가장 빈번히 사용되는 두 속성입니다.

### 1) `textContent` (보안 안전)
* **특징**: 할당하는 값을 순수한 **"텍스트 문자열"** 로만 해석하여 렌더링합니다. HTML 태그가 섞여 있어도 태그 기능을 잃고 일반 글자(예: `&lt;h1&gt;`)로 안전하게 화면에 출력됩니다.
* **보안성**: 외부 악성 사용자가 작성한 스크립트 코드(`"<script>stealCookie()</script>"`)가 주입되어 브라우저 내에서 즉각 샐행되는 **크로스 사이트 스크립팅(XSS, Cross-Site Scripting)** 해킹 공격을 원천적으로 차단합니다. 따라서 단순 글자 교체 시에는 무조건 `textContent` 사용이 철칙입니다.

### 2) `innerHTML` (주의해서 사용)
* **특징**: 할당하는 값에 포함된 HTML 코드를 브라우저가 직접 파싱하여 **실제 요소 노드로 변환** 해 줍니다. 
* **위험성**: 신뢰할 수 없는 사용자의 인풋값(게시판 글 등)을 `innerHTML`로 여과 없이 렌더링하면 보안 취약점이 뚫립니다. 
* **실무 대책**: 개발자가 정적 데이터(자체 JSON 등)를 바탕으로 신뢰할 수 있는 마크업 덩어리를 그릴 때만 제한적으로 활용해야 합니다.

---

## 3. DOM 노드 동적 제어 (생성·주입·삭제)
화면에 없는 요소를 메모리 영역에 임시 생성하고 문서에 부착시키는 정교한 수명 주기 조작법입니다.

```
                  ┌──────────────────────┐
                  │    Parent Element    │
                  └──────────┬───────────┘
            ┌────────────────┴────────────────┐
            ▼ (prepend: 맨 앞에 주입)          ▼ (appendChild: 맨 뒤에 주입)
   ┌──────────────────┐               ┌──────────────────┐
   │ New Child Element│               │ New Child Element│
   └──────────────────┘               └──────────────────┘
```

### 1) `document.createElement("태그명")`
* 지정된 태그를 가지는 비어있는 요소 객체를 메모리상에 새로 생성합니다. 
* 예: `const card = document.createElement("article");`

### 2) 노드 부착 (주입)
* **`부모.appendChild(자식)`**: 지정한 자식 요소를 부모 컨테이너의 **맨 마지막 자식 노드**로 덧붙여 넣습니다.
* **`부모.prepend(자식)`**: 지정한 자식 요소를 부모 컨테이너의 **맨 첫 번째 자식 노드** 로 끼워 넣습니다.

### 3) 노드 삭제
* **`요소.remove()`**: 문서상에서 해당 요소를 깨끗하게 즉각 삭제 처리합니다.

---

## 4. 데이터 기반 반복 렌더링 (Iterative Rendering)
실무에서 가장 많이 쓰이는 구조로, 정형화된 데이터 배열(JSON 모사)을 순회하며 요소를 동적으로 대량 생산해 내는 구조입니다.

```javascript
// 1. 백엔드에서 날아온 가상의 특징 데이터 배열
const features = [
  { icon: "[Fast]", title: "빠른 속도", desc: "최적화된 성능" },
  { icon: "[Secure]", title: "안전한 보안", desc: "데이터 암호화" }
];

const featuresContainer = document.querySelector(".features-grid");

// 2. 반복 루프와 구조분해 할당(Destructuring) 결합
features.forEach(({ icon, title, desc }) => {
  // 메모리에 비어있는 article 노드 조립
  const card = document.createElement("article");
  card.className = "feature-card";
  
  // 백틱 템플릿 리터럴로 내용 구조 형성 (신뢰할 수 있는 정적 데이터이므로 innerHTML 허용)
  card.innerHTML = `
    <span class="icon">${icon}</span>
    <h3>${title}</h3>
    <p>${desc}</p>
  `;
  
  // 컨테이너 맨 뒤에 조립 완료된 카드 차곡차곡 부착
  featuresContainer.appendChild(card);
});
```

---

## 5. 실습 
대시보드 알림 메시지 센터를 구현하여, 알림 수신 시 `document.createElement`로 요소를 동적 생성하고, `prepend()`로 최신 알림을 상단에 삽입합니다. 또한 대량의 초기 데이터를 렌더링할 때 성능을 비약적으로 개선해 주는 실무 표준 최적화 API인 **`DocumentFragment`** 를 활용하는 성능 지향형 코드를 매핑합니다.

* **적용 파일**: `main_04.js`
* **적용 대상**: 대시보드 실시간 알림 목록
```javascript
const noticeContainer = document.querySelector("#dynamic-grid");
const noticesData = [
  { title: "시스템 업데이트", desc: "서버 점검이 완료되었습니다." },
  { title: "새 메시지", desc: "이영희 님이 답글을 남겼습니다." }
];

// 1. 메모리 상에 가상의 빈 보관함(DocumentFragment)을 생성
const fragment = document.createDocumentFragment();

noticesData.forEach(({ title, desc }) => {
  const noticeCard = document.createElement("article");
  noticeCard.className = "feature-card";
  noticeCard.innerHTML = `<h3>${title}</h3><p>${desc}</p>`;
  
  // 브라우저 돔에 직접 넣는 것이 아닌, 가상 보관함에 1차 누적 (Reflow 발생 0회)
  fragment.appendChild(noticeCard);
});

// 2. 단 1회의 리플로우(Reflow)로 대량의 카드를 화면에 즉시 장착
noticeContainer.appendChild(fragment);
```

# [7교시]

# 이벤트 리스너 기초

## 1. 학습 목표
* DOM 트리가 브라우저 메모리에 파싱 및 구축 완료되는 안전한 시작 시점을 잡아내는 `DOMContentLoaded` 이벤트의 중요성을 이해합니다.
* HTML 마크업 내부에 직접 자바스크립트를 섞어 적는 구식 인라인 이벤트 속성(`onclick`)의 심각한 단점을 파악하고, `addEventListener` 표준 등록 방식과 핸들러 함수 분리 아키텍처를 적용합니다.
* 이벤트가 발생했을 때 웹 브라우저가 콜백 인자로 자동 전달해 주는 이벤트 객체(`event`)의 중요 속성인 `target`과 `currentTarget`을 파악합니다.

---

## 2. 안전한 스크립트 실행의 보증수표: DOMContentLoaded
만약 HTML 파일 헤더 부분에서 스크립트를 로드했는데 스크립트 내부에 `document.querySelector(".hero")`와 같이 DOM을 탐색하는 코드가 들어있다면, 브라우저가 아직 바디의 해당 태그를 파싱하기도 전이므로 **`null` 참조 에러**가 발생합니다.

이를 근본적으로 안전하게 방어하기 위해 전체 스크립트를 감싸는 진입점 이벤트 리스너를 설계합니다.

```javascript
// DOM 파싱이 완전히 완료되어 태그 메모리 적재가 끝난 즉시 호출됩니다. (이미지 로딩 완료는 대기하지 않아 로딩이 빠름)
document.addEventListener("DOMContentLoaded", () => {
  console.log("DOM 트리가 준비되었습니다. 자바스크립트 초기화 로직을 구동합니다.");
  init(); // 전체 이벤트 바인딩 및 렌더링 구동 함수
});
```

---

## 3. onclick 속성 금지 vs addEventListener 권장

### 1) 구식 방식 (완전 금지)
`<button onclick="handleBtnClick()">제출</button>`
* **단점**: HTML 마크업과 JS 제어 로직이 더럽게 뒤섞여 협업이 불가능합니다. 또한 하나의 버튼에 단 하나의 동작만 바인딩할 수 있어 다른 개발자가 쓴 동작이 기존 스크립트를 덮어쓰는 대형 버그를 일으킵니다.

### 2) 모던 addEventListener 표준 (실무 표준)
* **장점**: HTML은 뼈대만 두고 JS 파일 내부에서 완벽하게 제어합니다(관심사 분리). 하나의 요소에 여러 개의 다른 리스너를 등록해 두어도 모두 정상적으로 동시 구동됩니다.
* **함수 분리 규칙**: 익명 함수(람다)를 직접 대입하기보다, **이름이 명확히 지어진 외부 핸들러 함수** 로 분리하여 바인딩하는 것이 메모리 최적화 및 유지보수, 이벤트 해제(`removeEventListener`)에 필수적입니다.

```javascript
// 핸들러 함수 분리 아키텍처
const handleMenuToggle = (event) => {
  const menu = document.querySelector(".nav-menu");
  menu.classList.toggle("open");
};

// 바인딩
menuBtn.addEventListener("click", handleMenuToggle);
```

---

## 4. 이벤트 객체 (Event Object) 와 event.target
사용자가 마우스를 클릭하거나 키보드를 입력하면, 브라우저는 발생한 이벤트에 대한 온갖 메타데이터(클릭 좌표, 입력된 키 값, 클릭된 요소 등)를 담은 **이벤트 객체(`e` 또는 `event`)**를 생성하여 핸들러 함수의 첫 번째 인자로 던져줍니다.

* **`event.target`**: 실제 물리적으로 **마우스 클릭 이벤트가 직접 일어난 최하단 요소 노드**를 가리킵니다. (예: 버튼 글씨를 감싸고 있는 `<span>` 태그를 클릭했다면 target은 `<span>`이 됨)
* **`event.currentTarget`**: 이벤트 리스너가 **실제로 바인딩되어 동작을 대기하고 있던 요소 노드**를 가리킵니다. (예: 버튼 자체에 addEventListener를 걸었다면 target이 span이더라도 currentTarget은 버튼 요소가 됨)

---

## 5. 실습 미션 적용 (이론 적용 코드)
대중적인 쇼핑몰의 "장바구니 담기" 시스템을 통해 복수 이벤트 등록과 타깃 노드 분석을 구현합니다.

* **적용 파일**: `main_05.js`
* **적용 대상**: 장바구니 추가 버튼 및 카운터
```javascript
// DOM 요소 선택
const addCartBtn = document.querySelector("#btn-add-cart");
const cartCountEl = document.querySelector("#cart-count");

// 기능 A: 장바구니 숫자 증가
const handleCartCount = () => {
  const currentCount = parseInt(cartCountEl.textContent ?? "0", 10);
  cartCountEl.textContent = currentCount + 1;
};

// 기능 B: 이벤트 객체 디버깅 로그 출력
const handleDebugLog = (event) => {
  console.log("실제 클릭한 지점 (event.target):", event.target);
  console.log("이벤트가 바인딩된 버튼 (event.currentTarget):", event.currentTarget);
};

// DOMContentLoaded 초기화 루프 안에서 안전하게 두 기능을 버튼 하나에 바인딩
document.addEventListener("DOMContentLoaded", () => {
  addCartBtn.addEventListener("click", handleCartCount);
  addCartBtn.addEventListener("click", handleDebugLog);
});
```

---

## 6. 학습 정리 체크리스트
- [ ] HTML 바디 하단에 스크립트를 로드하는 방식이 있어도 `DOMContentLoaded` 리스너를 명시적으로 쓰는 설계적 이유는 무엇인가?
- [ ] HTML 파일 안의 태그에 `onclick="..."` 속성을 사용하면 왜 유지보수가 파괴되는지 서술할 수 있는가?
- [ ] 이벤트 핸들러 함수에 매개변수로 명시하는 `event` 변수는 누가 채워서 넘겨주는가?
- [ ] 버튼 안의 폰트 아이콘(`<i>` 등)을 마우스로 조준하여 눌렀을 때 `event.target`과 `event.currentTarget`이 각각 지목하는 HTML 요소는 어떻게 다른가?


# [8교시]

# 이벤트 전파(버블링)와 이벤트 위임

## 1. 학습 목표
* HTML 트리 구조에서 자식 노드에 발생한 이벤트가 상위 부모 노드들로 꼬리를 물고 연쇄 전파되는 이벤트 버블링(Event Bubbling)의 개념을 파악합니다.
* 이벤트 전파를 인위적으로 중단시키는 `event.stopPropagation()`의 문법적 역할과 이를 실무에서 오남용할 시 발생하는 부작용을 이해합니다.
* 다수의 동일 유형 요소에 개별 이벤트 리스너를 바인딩하지 않고, 공통 상위 부모 1곳에만 리스너를 결합해 이벤트를 병합 처리하는 실무 최적화 아키텍처인 **이벤트 위임(Event Delegation)** 을 마스터합니다.

---

## 2. 이벤트 버블링 (Event Bubbling) 이란?
HTML의 특정 태그(예: `<span>`)를 클릭하면, 이벤트는 단순히 그 태그에만 머무르지 않고 부모(`<li>`) -> 상위 부모(`<nav>`) -> 더 상위 부모(`<body>`) 순으로 거슬러 올라가며 전파됩니다. 마치 물속의 거품(Bubble)이 위로 둥둥 떠오르는 모습과 흡사하여 **버블링** 이라 명명되었습니다.

```
[HTML 트리 구조]
   document  (최상위 조상)
      ▲
     body
      ▲
    section
      ▲
     div  (.gallery)          <-- 부모 컨테이너
      ▲
   article  (.gallery-item)   <-- 중간 자식
      ▲
     img  [실제 클릭 target]   <-- 마우스가 물리적으로 가리킨 표적 노드
```

> [!NOTE]
> 대부분의 웹 브라우저 이벤트(click, keydown 등)는 기본 설정이 버블링입니다. (단, focus, blur, mouseenter, mouseleave 등은 전파되지 않는 특수 이벤트입니다.)

---

## 3. 이벤트 전파 차단: `event.stopPropagation()`
* **역할**: 핸들러 함수 내부에서 이 메서드를 호출하면, 현재 노드 이후로 이벤트가 상위 부모로 거슬러 올라가지 않도록 물리적인 흐름을 강제 중단시킵니다.
* **실무적 주의사항**: 버블링을 강제로 막아버리면 문서 전체 영역에서 글로벌 이벤트를 감지(예: 바깥쪽 아무 데나 누르면 팝업창 닫히는 기능 등)할 때 버블링이 끊겨 오작동하게 됩니다. 따라서 특수한 팝업 버튼 등 예외적인 겹침 처리 상황을 제외하고는 **가급적 `stopPropagation()`의 무분별한 사용은 강하게 지양**해야 합니다.

---

## 4. 실무 성능의 핵심: 이벤트 위임 (Event Delegation)

만약 갤러리 썸네일 이미지가 1,000장 존재할 때 이미지 클릭 시 확대 팝업을 띄우고자 한다면, 1,000개의 `img` 마다 일일이 리스너를 다는 행위는 **메모리를 엄청나게 낭비** 하고 브라우저 성능을 바닥으로 끌어내립니다.

또한, 자바스크립트로 신규 이미지를 동적으로 1장 추가할 때마다 리스너 등록 코드를 매번 다시 실행해주어야 하는 지옥 같은 유지보수 장벽이 발생합니다.

### 해결책: 이벤트 위임 (Event Delegation)
자식들의 이벤트가 어차피 부모로 버블링되는 성격을 역이용하여, **부모 컨테이너 1곳에만 단 하나의 `addEventListener`를 등록** 해 놓습니다. 그리고 클릭된 대상(`event.target`)이 우리가 목표한 자식 요소를 가리키고 있는지 `closest()`로 필터링하여 총괄 수거하는 기법입니다.

```javascript
// 이벤트 위임 모범 예제
const galleryContainer = document.querySelector(".gallery-grid");

galleryContainer.addEventListener("click", (event) => {
  // 클릭된 대상(event.target)에서 가장 가까운 .gallery-item 조상을 역추적
  const clickedCard = event.target.closest(".gallery-item");
  
  // 1. 갤러리 카드 밖(빈 틈 등)을 누른 경우 즉각 스킵 처리 (방어 코드)
  if (!clickedCard) return;
  
  // 2. 갤러리 카드 내부인 경우 정상 구동
  const imgTitle = clickedCard.dataset.title;
  console.log("선택한 작품 정보:", imgTitle);
});
```

---

## 5. 실습 미션 적용 (이론 적용 코드)
대화형 FAQ 아코디언 컴포넌트의 부모 목록 컨테이너(`.faq-container`)에 단 하나의 클릭 이벤트 리스너를 위임 부착하고, 질문 제목 클릭 시 본문 내용(`.faq-answer`)을 펼치는 로직을 매핑합니다.

* **적용 파일**: `day02_javascript/06_event_delegation/main_06.js`
* **적용 대상**: FAQ 아코디언 목록
```javascript
const faqContainer = document.querySelector(".faq-container");

// 부모 컨테이너에 단 하나의 리스너만 매핑하여 위임 처리
faqContainer.addEventListener("click", (event) => {
  // closest를 이용해 클릭된 대상이 FAQ 아이템 조상인지 확인
  const faqItem = event.target.closest(".faq-item");
  if (!faqItem) return; // 빈 여백 클릭 방어

  // 질문 헤더 영역(.faq-question)을 정확히 지향하여 클릭했는지 확인
  const isQuestion = event.target.closest(".faq-question");
  if (isQuestion) {
    faqItem.classList.toggle("active");
  }
});
```

---

## 6. 학습 정리 체크리스트
- [ ] 자식 요소를 클릭했는데 부모 요소에 걸어놓은 클릭 이벤트 리스너가 호출되는 웹 브라우저의 전파 현상을 무엇이라 하는가?
- [ ] `event.stopPropagation()`을 실무 프로젝트에서 무분별하게 남발하면 어떤 부작용(예: 팝업 닫기 실패 등)이 발생할 수 있는가?
- [ ] 이벤트 위임을 적용했을 때, 개별 동적 요소에 리스너를 매번 재등록하지 않아도 무리 없이 감지할 수 있는 구동 원리를 버블링과 연계해 설명할 수 있는가?
- [ ] `closest(".gallery-item")`를 필터 방어 조건문(`if (!item) return;`)과 함께 결합하여 클릭 대상 오인식 에러를 회피할 수 있는가?


# 폼 이벤트와 입력 유효성 검사

## 1. 학습 목표
* 폼 제출 시 웹 브라우저가 화면을 강제로 새로고침해 버리는 기본 빌트인 동작의 원리를 파악하고, 이를 방지하기 위한 `event.preventDefault()`의 용도를 이해합니다.
* 사용자가 자판을 입력하는 즉시 감지하여 실시간 유효성 피드백을 제공하는 `input` 이벤트 제어 기법을 습득합니다.
* 값이 비어있거나 특정 DOM 객체가 존재하지 않을 때 스크립트 실행 오류로 뻗어 버리는 버그를 예방하기 위한 **옵셔널 체이닝(`?.`)** 및 **널 병합 연산자(`??`)** 활용을 마스터합니다.

---

## 2. preventDefault() 와 submit 제어
HTML의 `<form>` 태그는 내부의 전송 버튼을 누르면 action 경로로 HTTP 패킷을 쏘고 화면 전체를 강제 새로고침하는 고유의 기본 동작을 내장하고 있습니다.

하지만 현대 싱글 페이지 애플리케이션(SPA)이나 모던 동적 웹 개발에서는 화면 깜빡임 없이 비동기 통신(Fetch API 등)으로 백엔드 데이터를 수신하므로, 이 구식 브라우저 화면 새로고침 동작을 묶어두어야만 합니다.

```javascript
const handleFormSubmit = (event) => {
  // 브라우저의 양식 전송 + 새로고침 기본 동작을 정지시킵니다.
  event.preventDefault();
  
  // 이후 안전하게 자바스크립트 유효성 검사와 비동기 전송 처리 진행
  console.log("새로고침 없이 안전하게 폼 제출 데이터를 수집했습니다.");
};

form.addEventListener("submit", handleFormSubmit);
```

---

## 3. 실시간 유효성 검사: `input` 이벤트
* **`change` 이벤트**: 포커스가 입력창 밖으로 완전히 빠져나가거나 엔터를 쳐서 입력이 '완료'된 시점에만 딱 한 번 실행됩니다.
* **`input` 이벤트**: 사용자가 키보드를 1글자 칠 때마다 즉각적으로 반응하여 실행됩니다. 비밀번호 보안 강도 실시간 바, 이메일 형식 적합성 검사 시 뛰어난 즉시 피드백 효과를 제공합니다.

---

## 4. 안전한 변수 처리: 옵셔널 체이닝(?.) & 널 병합(??)
2026년 기준 실무 스크립팅에서 **"절대 빼놓을 수 없는 예외 회피 표준 문법"**입니다.

### 1) 옵셔널 체이닝 (`?.`)
* **정의**: 참조한 대상이 `null` 또는 `undefined`이면 에러를 뿜지 않고 즉시 스크립트 실행을 우회하며 `undefined` 값을 안전하게 반환합니다.
* **용도**: 입력 폼에 특정 필드가 누락되었거나 동적 DOM이 임시 소멸하여 찾을 수 없는 상태에서 하위 `.value`나 속성을 건드렸을 때 스크립트 전체가 에러로 터지는 버그를 원천 봉쇄합니다.
* **비교**:
  ```javascript
  // 과거 구식
  const value = (emailInput !== null && emailInput !== undefined) ? emailInput.value : undefined;
  // 모던 사양
  const value = emailInput?.value;
  ```

### 2) 널 병합 연산자 (`??`)
* **정의**: 좌측 피연산자의 값이 오직 `null` 또는 `undefined`일 때만 우측의 대체 기본값을 반환합니다. (기존 단축 평가 `||`가 `0` 이나 빈 문자열 `""`까지 거짓으로 취급해 기본값으로 덮어써 버리던 논리 왜곡 문제를 완벽하게 수정)
* **결합**: 옵셔널 체이닝과 결합하여 안전한 대체 기본값 처리에 활용합니다.
  ```javascript
  // emailInput이 없거나 value가 null이면 빈 문자열("")을 안전하게 보장
  const email = emailInput?.value ?? "";
  ```

---

## 5. 실습 미션 적용 (이론 적용 코드)
회원가입 Form에 대해 양식 전송 시의 강제 새로고침을 차단하고, `input` 이벤트를 통해 실시간으로 검증하며, 유효성 검사 통과 시 입력받은 정보를 URL 파라미터로 붙여 가입 완료 페이지(`welcome_07.html`)로 안전하게 전달합니다. 가입 완료 페이지의 스크립트(`welcome_07.js`)는 `URLSearchParams` API를 사용하여 쿼리 스트링을 파싱하고, 수집된 데이터를 화면에 출력할 때 옵셔널 체이닝(`?.`) 및 널 병합(`??`)을 적용하여 예외 방어를 구현합니다.

* **적용 파일**: `day02_javascript/07_form_validation/main_07.js` & `welcome_07.js`
* **적용 대상**: 회원가입 완료 리다이렉션 및 가입자 정보 파싱 화면 출력
```javascript
// 1. main_07.js: 유효성 검사 성공 시 리다이렉션 전송
if (isFormValid) {
  // 안전하게 한글 및 특수문자를 인코딩하여 파라미터 조립
  const redirectUrl = `welcome_07.html?name=${encodeURIComponent(nameVal)}&email=${encodeURIComponent(emailVal)}&subject=${encodeURIComponent(subjectVal)}`;
  
  // 1초 후 가입 환영 페이지로 안전하게 정보 이동 처리
  setTimeout(() => {
    window.location.href = redirectUrl;
  }, 1000);
}

// 2. welcome_07.js: 가입 완료 페이지에서의 안전 데이터 수거 및 출력
const urlParams = new URLSearchParams(window.location.search);

// 옵셔널 체이닝(?.)과 널 병합(??)의 결합을 통해, 유실되거나 null인 정보에 대해 안전하게 기본값 대입 방어
const userName = urlParams?.get("name") ?? "익명 회원";
const userEmail = urlParams?.get("email") ?? "미등록 이메일";

// DOM 바인딩 처리
document.getElementById("welcome-name").textContent = userName;
document.getElementById("welcome-email").textContent = userEmail;
```

---

## 6. 학습 정리 체크리스트
- [ ] `form` 요소에 submit 동작이 가해질 때 브라우저 화면이 리로드되는 사유와 이를 차단하는 함수가 무엇인지 아는가?
- [ ] 입력값을 타이핑하는 족족 감지하는 `input` 이벤트와 포커스를 잃어야 가동되는 `change` 이벤트의 차이를 아는가?
- [ ] 옵셔널 체이닝(`?.`)을 거치지 않고 `null` 요소의 `.value`를 참조하려 할 때 콘솔에 발생하는 치명적 에러 유형은 무엇인가?
- [ ] 널 병합 연산자(`??`)가 기존의 `||` 논리 연산자 결합 구조보다 실무 폼 데이터 빈 문자열 감지 처리에서 탁월한 이유를 설명할 수 있는가?
